In [1]:
# install
!pip install yfinance alpaca-py plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 3.3 MB/s eta 0:00:00


In [2]:
# import
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

In [3]:
# connect
API_KEY    = "key"
API_SECRET = "secret"

client = TradingClient(API_KEY, API_SECRET, paper=True)
print(f"Status: {client.get_account().status}")
print(f"Cash:   ${float(client.get_account().cash):,.2f}")

Status: AccountStatus.ACTIVE
Cash:   $4,088.99


In [4]:
# ticker
def get_sp500_tickers():
    # iShares first
    try:
        holdings = pd.read_csv(
            'https://www.ishares.com/us/products/239726/'
            'ISHARES-CORE-SP-500-ETF/1467271812596.ajax'
            '?fileType=csv&fileName=IVV_holdings'
            '&dataType=fund',
            skiprows=9
        )
        holdings = holdings.dropna(subset=['Ticker'])
        holdings = holdings[
            holdings['Asset Class'] == 'Equity'
        ]
        tickers  = holdings['Ticker'].tolist()
        tickers  = [t.replace('.', '-') for t in tickers
                    if isinstance(t, str)
                    and t != 'nan']
        if len(tickers) > 400:
            print(f"iShares: found {len(tickers)} tickers")
            return tickers
    except Exception as e:
        print(f"iShares failed: {e}")

    # Wikipedia second
    try:
        sp500   = pd.read_html(
            'https://en.wikipedia.org/wiki/'
            'List_of_S%26P_500_companies'
        )[0]
        tickers = sp500['Symbol'].tolist()
        tickers = [t.replace('.', '-') for t in tickers]
        if len(tickers) > 400:
            print(f"Wikipedia: found {len(tickers)} tickers")
            return tickers
    except Exception as e:
        print(f"Wikipedia failed: {e}")

    # SPDR last
    try:
        holdings = pd.read_excel(
            'https://www.ssga.com/us/en/intermediary/'
            'etfs/library-content/products/fund-data/'
            'etfs/us/holdings-daily-us-en-spy.xlsx',
            skiprows=4
        )
        holdings = holdings.dropna(subset=['Ticker'])
        tickers  = holdings['Ticker'].tolist()
        tickers  = [str(t).replace('.', '-')
                    for t in tickers
                    if str(t) != 'nan']
        if len(tickers) > 400:
            print(f"SPDR: found {len(tickers)} tickers")
            return tickers
    except Exception as e:
        print(f"SPDR failed: {e}")

    raise Exception("All sources failed — "
                    "check your internet connection")

tickers = get_sp500_tickers()

iShares failed: Error tokenizing data. C error: Expected 1 fields in line 16, saw 4

Wikipedia failed: HTTP Error 403: Forbidden
SPDR: found 505 tickers


In [5]:
# download and clean
print("Downloading data... takes 2-3 minutes")
data = yf.download(
    tickers,
    start='2018-01-01',
    auto_adjust=True,
    progress=True
)['Close']

data = data.dropna(thresh=int(len(data) * 0.90), axis=1)
print(f"Stocks after cleaning: {data.shape[1]}")

[********************  42%                       ]  213 of 505 completedERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 2602335D"}}}
[*********************100%***********************]  505 of 505 completed
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['2602335D', '-']: YFTzMissingError('possibly delisted; no timezone found')


Stocks after cleaning: 477


In [6]:
# liquidity filter
volume = yf.download(
    tickers,
    start='2018-01-01',
    auto_adjust=True,
    progress=False
)['Volume']

avg_dollar_vol = volume.mean() * data.mean()
liquid         = avg_dollar_vol[
    avg_dollar_vol > 10_000_000
].index
data           = data[[c for c in liquid
                        if c in data.columns]]
print(f"Stocks after liquidity filter: {data.shape[1]}")

ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['2602335D', '-']: YFTzMissingError('possibly delisted; no timezone found')


Stocks after liquidity filter: 477


In [7]:
# mean reversion signal
# Z-score over 20 days
# Buy the 10 most oversold stocks (Z < -2)

def zscore(prices, window=20):
    mean = prices.rolling(window).mean()
    std  = prices.rolling(window).std()
    return (prices - mean) / std

z = zscore(data)

def mean_rev_weights(z_scores, top_n=10,
                     z_threshold=-1.5):
    # Only buy stocks that are genuinely oversold
    # Z below threshold AND ranked most oversold
    oversold = (z_scores < z_threshold).astype(int)
    ranks    = z_scores.rank(axis=1, ascending=True)
    in_top   = (ranks <= top_n).astype(int)
    signal   = oversold * in_top
    weights  = signal.div(
        signal.sum(axis=1), axis=0
    ).fillna(0)
    return weights, signal

weights, signal = mean_rev_weights(z)

# current picks
latest_z  = z.iloc[-1]
latest_w  = weights.iloc[-1]
picks     = latest_w[latest_w > 0].sort_values(
    ascending=False
)

print(f"=== CURRENT OVERSOLD PICKS ===")
print(f"Selected from {data.shape[1]} stocks\n")

if len(picks) == 0:
    print("No stocks oversold enough right now — "
          "holding cash")
else:
    for t, w in picks.items():
        print(f"  {t}: {w:.0%} | Z-score: "
              f"{latest_z[t]:.2f}")

=== CURRENT OVERSOLD PICKS ===
Selected from 477 stocks

  APA: 10% | Z-score: -2.06
  BG: 10% | Z-score: -1.97
  BKR: 10% | Z-score: -1.94
  HAL: 10% | Z-score: -2.26
  LDOS: 10% | Z-score: -2.26
  LVS: 10% | Z-score: -1.88
  OXY: 10% | Z-score: -2.01
  PSX: 10% | Z-score: -2.11
  SLB: 10% | Z-score: -2.53
  VRSN: 10% | Z-score: -1.93


In [9]:
# Z-Score Rankings
import plotly.graph_objects as go

# today's z-scores and ranks
latest_z     = z.iloc[-1].dropna()
latest_ranks = latest_z.rank(ascending=True)
latest_picks = weights.iloc[-1]

# summary table
summary = pd.DataFrame({
    'Z-Score':  latest_z,
    'Rank':     latest_ranks,
    'Price':    data.iloc[-1],
    'MA20':     data.rolling(20).mean().iloc[-1],
    'Weight':   latest_picks
}).dropna()
summary = summary.sort_values('Z-Score')
summary['Selected'] = summary['Weight'] > 0

# Add 1-day and 5-day returns
ret1d = data.pct_change(1).iloc[-1] * 100
ret5d = data.pct_change(5).iloc[-1] * 100
summary['1D Return %'] = ret1d
summary['5D Return %'] = ret5d

print("=" * 70)
print(f"  Z-SCORE RANKINGS — "
      f"{data.index[-1].strftime('%B %d, %Y')}")
print("=" * 70)

selected    = summary[summary['Selected']]
oversold    = summary[summary['Z-Score'] < -1.5]
overbought  = summary[summary['Z-Score'] > 1.5]
neutral     = summary[
    (summary['Z-Score'] >= -1.5) &
    (summary['Z-Score'] <= 1.5)
]

print(f"\n  Oversold  (Z < -1.5): {len(oversold)} stocks")
print(f"  Neutral   (-1.5 to +1.5): "
      f"{len(neutral)} stocks")
print(f"  Overbought (Z > +1.5): "
      f"{len(overbought)} stocks")
print(f"  Currently holding: "
      f"{len(selected)} stocks\n")

print(f"  {'Rank':<6} {'Ticker':<8} {'Z-Score':>8} "
      f"{'Price':>8} {'20d MA':>8} "
      f"{'1D Ret':>8} {'5D Ret':>8} {'Status':<15}")
print(f"  {'-'*6} {'-'*8} {'-'*8} "
      f"{'-'*8} {'-'*8} "
      f"{'-'*8} {'-'*8} {'-'*15}")

# most oversold first
for rank, (ticker, row) in enumerate(
    summary.iterrows(), 1
):
    z_score  = row['Z-Score']
    price    = row['Price']
    ma20     = row['MA20']
    ret1     = row['1D Return %']
    ret5     = row['5D Return %']
    selected = row['Selected']

    if z_score < -1.5:
        status = '◄ HOLDING' if selected \
                 else 'OVERSOLD'
    elif z_score > 1.5:
        status = 'OVERBOUGHT'
    else:
        status = 'neutral'

    # show extreme stocks
    if abs(z_score) < 1.0 and not selected:
        continue

    z_arrow = '▼' if z_score < 0 else '▲'
    print(f"  {rank:<6} {ticker:<8} "
          f"{z_arrow}{abs(z_score):>7.2f}  "
          f"${price:>7.2f} "
          f"${ma20:>7.2f} "
          f"{ret1:>+7.1f}% "
          f"{ret5:>+7.1f}% "
          f"{status:<15}")

    if rank >= 30:
        break

# Chart 1 — Z-score bar chart (most extreme stocks)
top20extreme = pd.concat([
    summary.head(20),
    summary.tail(20)
])

colors = []
for _, row in top20extreme.iterrows():
    if row['Selected']:
        colors.append('#534AB7')
    elif row['Z-Score'] < -1.5:
        colors.append('#D85A30')
    elif row['Z-Score'] > 1.5:
        colors.append('#EF9F27')
    else:
        colors.append('#888780')

fig = go.Figure(go.Bar(
    x=top20extreme.index,
    y=top20extreme['Z-Score'],
    marker_color=colors,
    text=[f"{v:.2f}"
          for v in top20extreme['Z-Score']],
    textposition='outside'
))

fig.add_hline(
    y=-1.5,
    line_dash='dash',
    line_color='#534AB7',
    annotation_text='buy threshold (-1.5)'
)
fig.add_hline(
    y=1.5,
    line_dash='dash',
    line_color='#EF9F27',
    annotation_text='overbought (+1.5)'
)
fig.add_hline(
    y=0,
    line_color='#888780',
    line_width=0.5
)

fig.update_layout(
    title='Z-Scores — Most Oversold & Overbought  '
          '(purple = holding, red = oversold not selected, '
          'amber = overbought)',
    yaxis_title='Z-Score (std deviations from 20d mean)',
    xaxis_title='Stock',
    height=500,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig.show()

# Chart 2 — Z-score distribution across all stocks
fig2 = go.Figure(go.Histogram(
    x=summary['Z-Score'],
    nbinsx=60,
    marker_color='#534AB7',
    opacity=0.8
))
fig2.add_vline(
    x=-1.5,
    line_dash='dash',
    line_color='#D85A30',
    annotation_text='buy zone'
)
fig2.add_vline(
    x=1.5,
    line_dash='dash',
    line_color='#EF9F27',
    annotation_text='overbought'
)
fig2.add_vline(
    x=0,
    line_color='#888780',
    line_width=1
)
fig2.update_layout(
    title='Z-Score Distribution — All 500 Stocks Today',
    xaxis_title='Z-Score',
    yaxis_title='Number of stocks',
    height=400,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig2.show()

# Chart 3 — holding's Z-score over last 30 days
if len(selected) > 0:
    held_tickers = selected.index.tolist()
    z_history    = z[held_tickers].tail(30)

    fig3 = go.Figure()
    colors_held = [
        '#534AB7','#1D9E75','#D85A30',
        '#EF9F27','#185FA5','#993556',
        '#3B6D11','#854F0B','#A32D2D','#0F6E56'
    ]
    for i, ticker in enumerate(held_tickers):
        fig3.add_trace(go.Scatter(
            x=z_history.index,
            y=z_history[ticker],
            name=ticker,
            line=dict(
                color=colors_held[
                    i % len(colors_held)
                ],
                width=2
            )
        ))

    fig3.add_hline(
        y=-1.5,
        line_dash='dash',
        line_color='#888780',
        annotation_text='buy threshold'
    )
    fig3.add_hline(
        y=0,
        line_color='#888780',
        line_width=0.5
    )
    fig3.update_layout(
        title='Z-Score History — '
              'Your Current Holdings (last 30 days)',
        yaxis_title='Z-Score',
        height=450,
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    fig3.show()

# summary stats
avg_z    = summary['Z-Score'].mean()
median_z = summary['Z-Score'].median()

print(f"\n  MARKET STRESS SUMMARY")
print(f"  {'─'*50}")
print(f"  Avg Z-score (all stocks):    {avg_z:+.2f}")
print(f"  Median Z-score:              {median_z:+.2f}")
print(f"  Most oversold:               "
      f"{summary.index[0]} "
      f"(Z = {summary['Z-Score'].iloc[0]:.2f})")
print(f"  Most overbought:             "
      f"{summary.index[-1]} "
      f"(Z = {summary['Z-Score'].iloc[-1]:.2f})")

if avg_z < -0.5:
    print(f"\n  ⚠ Market is broadly oversold — "
          f"high opportunity environment")
elif avg_z > 0.5:
    print(f"\n  ⚠ Market is broadly overbought — "
          f"fewer opportunities, be selective")
else:
    print(f"\n  ✓ Market is neutral — "
          f"normal mean reversion conditions")

if len(selected) == 0:
    print(f"\n  No stocks oversold enough today — "
          f"holding cash is correct")
else:
    print(f"\n  CURRENT HOLDINGS Z-SCORES")
    print(f"  {'─'*50}")
    for ticker, row in selected.iterrows():
        print(f"  {ticker:<8} Z = {row['Z-Score']:>+.2f} | "
              f"1D: {row['1D Return %']:>+.1f}% | "
              f"5D: {row['5D Return %']:>+.1f}%")

/tmp/ipykernel_1927/3561644432.py:21: FutureWarning:

The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

/tmp/ipykernel_1927/3561644432.py:22: FutureWarning:

The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



  Z-SCORE RANKINGS — June 17, 2026

  Oversold  (Z < -1.5): 34 stocks
  Neutral   (-1.5 to +1.5): 330 stocks
  Overbought (Z > +1.5): 113 stocks
  Currently holding: 10 stocks

  Rank   Ticker    Z-Score    Price   20d MA   1D Ret   5D Ret Status         
  ------ -------- -------- -------- -------- -------- -------- ---------------
  1      SLB      ▼   2.53  $  51.50 $  55.64    -3.0%    -7.2% ◄ HOLDING      
  2      LDOS     ▼   2.26  $ 111.68 $ 123.20    -1.7%    -7.9% ◄ HOLDING      
  3      HAL      ▼   2.26  $  36.48 $  39.75    -1.9%    -8.2% ◄ HOLDING      
  4      PSX      ▼   2.11  $ 168.76 $ 178.13    -1.9%    -7.1% ◄ HOLDING      
  5      APA      ▼   2.06  $  34.15 $  37.07    -0.3%   -10.1% ◄ HOLDING      
  6      OXY      ▼   2.01  $  53.57 $  56.96    -0.2%    -6.2% ◄ HOLDING      
  7      BG       ▼   1.97  $ 115.76 $ 124.27    -3.1%    -9.7% ◄ HOLDING      
  8      BKR      ▼   1.94  $  60.77 $  64.02    -1.2%    -3.6% ◄ HOLDING      
  9      VRSN     ▼   1.9

TypeError: object of type 'bool' has no len()

In [ ]:
# backtest
daily_returns = data.pct_change(fill_method=None)
port_returns  = (weights.shift(1) *
                 daily_returns).sum(axis=1)

spy     = yf.download('SPY', start='2018-01-01',
                       auto_adjust=True)['Close']
spy     = spy.squeeze()
spy_ret = spy.pct_change(fill_method=None).squeeze()
spy_cum = (1 + spy_ret).cumprod()
cum     = (1 + port_returns).cumprod()

def sharpe(r, rf=0.02):
    e = r - rf/252
    return float((e.mean() / e.std()) * np.sqrt(252))

def max_drawdown(r):
    c = (1 + r).cumprod()
    return float(((c - c.cummax()) / c.cummax()).min())

def cagr(r):
    c = (1 + r).cumprod()
    y = len(r) / 252
    return float(c.iloc[-1] ** (1/y) - 1)

def win_rate(r):
    return float((r > 0).sum() / len(r))

days_in_market = (weights.sum(axis=1) > 0).sum()
total_days     = len(weights)

print(f"Strategy CAGR:    {cagr(port_returns):.1%}")
print(f"SPY CAGR:         {cagr(spy_ret):.1%}")
print(f"Sharpe:           {sharpe(port_returns):.2f}")
print(f"Max Drawdown:     {max_drawdown(port_returns):.1%}")
print(f"Win Rate:         {win_rate(port_returns):.1%}")
print(f"Days in market:   {days_in_market/total_days:.0%}")
print(f"Best Day:         {port_returns.max():.1%}")
print(f"Worst Day:        {port_returns.min():.1%}")

In [ ]:
# charts
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        f'Mean Reversion vs SPY '
        f'(universe: {data.shape[1]} stocks)',
        'Drawdown',
        'Days With Active Positions'
    ),
    row_heights=[0.45, 0.25, 0.30],
    vertical_spacing=0.08
)

fig.add_trace(go.Scatter(
    x=cum.index, y=(cum-1)*100,
    name='Mean Reversion',
    line=dict(color='#534AB7', width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=spy_cum.index, y=(spy_cum-1)*100,
    name='SPY',
    line=dict(color='#888780', width=1.5, dash='dash')
), row=1, col=1)

dd = (cum / cum.cummax() - 1) * 100
fig.add_trace(go.Scatter(
    x=dd.index, y=dd,
    fill='tozeroy',
    line=dict(color='#D85A30', width=1),
    fillcolor='rgba(216,90,48,0.15)',
    name='Drawdown'
), row=2, col=1)

# Number of active positions over time
n_positions = signal.sum(axis=1)
fig.add_trace(go.Scatter(
    x=n_positions.index,
    y=n_positions,
    fill='tozeroy',
    line=dict(color='#534AB7', width=1),
    fillcolor='rgba(83,74,183,0.15)',
    name='# Positions'
), row=3, col=1)

fig.update_layout(
    height=850,
    title='Mean Reversion — S&P 500 Universe',
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig.update_yaxes(ticksuffix='%', row=1, col=1)
fig.update_yaxes(ticksuffix='%', row=2, col=1)
fig.show()

# Current Z-score distribution
z_today = z.iloc[-1].dropna().sort_values()
colors  = ['#534AB7' if t in picks.index
           else '#D85A30' if v < -1.5
           else '#888780'
           for t, v in z_today.items()]

# shows most extreme 40 stocks for readability
extreme = pd.concat([z_today.head(20), z_today.tail(20)])
fig2 = go.Figure(go.Bar(
    x=extreme.index,
    y=extreme.values,
    marker_color=['#534AB7' if t in picks.index
                  else '#D85A30' if v < -1.5
                  else '#1D9E75' if v > 1.5
                  else '#888780'
                  for t, v in extreme.items()]
))
fig2.add_hline(y=-1.5, line_dash='dash',
               line_color='#534AB7',
               annotation_text='buy threshold (-1.5)')
fig2.add_hline(y=1.5, line_dash='dash',
               line_color='#D85A30',
               annotation_text='overbought (+1.5)')
fig2.update_layout(
    title='Most Oversold & Overbought Stocks Today '
          '— purple = selected',
    yaxis_title='Z-Score',
    height=450,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig2.show()

In [10]:
# smart rebalance
positions     = client.get_all_positions()
port_value    = float(client.get_account().cash) + \
                sum([float(p.market_value) for p in positions])
current_holds = {p.symbol: float(p.market_value)
                 for p in positions}

if len(picks) == 0:
    print("No oversold stocks today — closing all "
          "positions and holding cash")
    for t in current_holds:
        client.close_position(t)
        print(f"  Closed {t}")
else:
    target = {t: port_value * 0.95 * w
              for t, w in picks.items()}

    print(f"Portfolio value: ${port_value:,.2f}")
    print(f"Deploying across {len(picks)} stocks\n")

    all_tickers = set(current_holds.keys()) | \
                  set(target.keys())

    print("=== CURRENT vs TARGET ===")
    for t in sorted(all_tickers):
        cur  = current_holds.get(t, 0)
        want = target.get(t, 0)
        diff = want - cur
        print(f"  {t}: have ${cur:,.0f} → "
              f"want ${want:,.0f} "
              f"({'buy' if diff>0 else 'SELL'} "
              f"${abs(diff):,.0f})")

    print("\n=== PLACING ORDERS ===")

    for t in all_tickers:
        cur  = current_holds.get(t, 0)
        want = target.get(t, 0)
        diff = want - cur
        if want == 0 and cur > 0:
            client.close_position(t)
            print(f"  SOLD {t} — Z-score normalized")
        elif diff < -500:
            client.submit_order(MarketOrderRequest(
                symbol=t,
                notional=round(abs(diff), 2),
                side=OrderSide.SELL,
                time_in_force=TimeInForce.DAY
            ))
            print(f"  Trimmed ${abs(diff):,.0f} of {t}")

    for t in all_tickers:
        cur  = current_holds.get(t, 0)
        want = target.get(t, 0)
        diff = want - cur
        if diff > 500:
            client.submit_order(MarketOrderRequest(
                symbol=t,
                notional=round(diff, 2),
                side=OrderSide.BUY,
                time_in_force=TimeInForce.DAY
            ))
            print(f"  Bought ${diff:,.0f} of {t}")
        elif abs(diff) <= 500:
            print(f"  {t}: skipping (too small)")

Portfolio value: $104,092.75
Deploying across 10 stocks

=== CURRENT vs TARGET ===
  ABT: have $9,126 → want $0 (SELL $9,126)
  APA: have $0 → want $9,889 (buy $9,889)
  AWK: have $9,601 → want $0 (SELL $9,601)
  BG: have $0 → want $9,889 (buy $9,889)
  BKR: have $0 → want $9,889 (buy $9,889)
  DLTR: have $10,559 → want $0 (SELL $10,559)
  GD: have $10,699 → want $0 (SELL $10,699)
  HAL: have $0 → want $9,889 (buy $9,889)
  HSY: have $8,871 → want $0 (SELL $8,871)
  JNJ: have $9,739 → want $0 (SELL $9,739)
  LDOS: have $0 → want $9,889 (buy $9,889)
  LVS: have $0 → want $9,889 (buy $9,889)
  OXY: have $0 → want $9,889 (buy $9,889)
  PM: have $11,329 → want $0 (SELL $11,329)
  PSX: have $0 → want $9,889 (buy $9,889)
  RSG: have $9,810 → want $0 (SELL $9,810)
  SLB: have $0 → want $9,889 (buy $9,889)
  TKO: have $10,293 → want $0 (SELL $10,293)
  VRSN: have $0 → want $9,889 (buy $9,889)
  WMB: have $9,975 → want $0 (SELL $9,975)

=== PLACING ORDERS ===
  SOLD RSG — Z-score normalized
  S

In [11]:
# Daily Summary Cell
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

print("=" * 55)
print(f"  PORTFOLIO SUMMARY")
print(f"  {datetime.now().strftime('%B %d, %Y %I:%M %p')}")
print("=" * 55)

# Account Overview
account    = client.get_account()
cash       = float(account.cash)
equity     = float(account.equity)
last_eq    = float(account.last_equity)
day_pl     = equity - last_eq
day_pl_pct = (day_pl / last_eq) * 100

print(f"\n  ACCOUNT")
print(f"  {'Total equity':<25} ${equity:>12,.2f}")
print(f"  {'Cash':<25} ${cash:>12,.2f}")
print(f"  {'Invested':<25} ${equity-cash:>12,.2f}")
print(f"  {'Today P&L':<25} "
      f"{'▲' if day_pl >= 0 else '▼'}"
      f" ${abs(day_pl):>10,.2f} "
      f"({day_pl_pct:+.2f}%)")

# Positions
positions = client.get_all_positions()

if not positions:
    print("\n  No open positions — fully in cash")
else:
    print(f"\n  POSITIONS ({len(positions)} stocks)\n")
    print(f"  {'Ticker':<8} {'Shares':>7} {'Entry':>9} "
          f"{'Current':>9} {'Value':>10} "
          f"{'Day %':>8} {'Total %':>8}")
    print(f"  {'-'*8} {'-'*7} {'-'*9} "
          f"{'-'*9} {'-'*10} "
          f"{'-'*8} {'-'*8}")

    total_unrealized = 0
    winners          = []
    losers           = []

    for p in sorted(positions,
                    key=lambda x: float(x.unrealized_plpc),
                    reverse=True):
        symbol        = p.symbol
        qty           = float(p.qty)
        entry         = float(p.avg_entry_price)
        current       = float(p.current_price)
        mkt_val       = float(p.market_value)
        unrealized    = float(p.unrealized_pl)
        unrealized_pc = float(p.unrealized_plpc) * 100
        day_pc        = float(p.change_today) * 100
        total_unrealized += unrealized

        # Arrow indicators
        day_arrow   = '▲' if day_pc >= 0 else '▼'
        total_arrow = '▲' if unrealized_pc >= 0 else '▼'

        print(f"  {symbol:<8} "
              f"{qty:>7.1f} "
              f"${entry:>8.2f} "
              f"${current:>8.2f} "
              f"${mkt_val:>9,.0f} "
              f"{day_arrow}{abs(day_pc):>6.2f}% "
              f"{total_arrow}{abs(unrealized_pc):>6.2f}%")

        if unrealized_pc > 0:
            winners.append((symbol, unrealized_pc))
        else:
            losers.append((symbol, unrealized_pc))

    # P&L Summary
    print(f"\n  {'─'*53}")
    unreal_arrow = '▲' if total_unrealized >= 0 else '▼'
    print(f"  {'Total unrealized P&L':<25} "
          f"{unreal_arrow} ${abs(total_unrealized):>9,.2f}")

    # Best and worst performers
    if winners:
        best = max(winners, key=lambda x: x[1])
        print(f"  {'Best position':<25} "
              f"{best[0]} (+{best[1]:.2f}%)")
    if losers:
        worst = min(losers, key=lambda x: x[1])
        print(f"  {'Worst position':<25} "
              f"{worst[0]} ({worst[1]:.2f}%)")

    # Portfolio health
    print(f"\n  PORTFOLIO HEALTH")
    n_up   = len(winners)
    n_down = len(losers)
    total  = len(positions)
    bar    = ('▲' * n_up) + ('▼' * n_down)
    print(f"  {n_up} up / {n_down} down  {bar}")

    avg_return = np.mean(
        [float(p.unrealized_plpc) * 100
         for p in positions]
    )
    print(f"  Avg position return:      {avg_return:+.2f}%")

    # Stop loss warnings
    warnings = [(p.symbol, float(p.unrealized_plpc)*100)
                for p in positions
                if float(p.unrealized_plpc) < -0.10]

    if warnings:
        print(f"\n  ⚠ STOP LOSS WARNINGS (down >10%)")
        for symbol, pct in warnings:
            print(f"    {symbol}: {pct:.2f}% — "
                  f"consider reviewing")

# Recent orders
print(f"\n  RECENT ORDERS (last 5)")
print(f"  {'─'*53}")

try:
    orders = client.get_orders(
        filter=dict(limit=5, status='all')
    )
    if not orders:
        print("  No recent orders")
    else:
        for o in orders:
            side   = o.side.value.upper()
            symbol = o.symbol
            status = o.status.value
            filled = o.filled_at
            amt    = (f"${float(o.notional):,.0f}"
                      if o.notional
                      else f"{float(o.qty)} shares")
            date   = (filled.strftime('%b %d %I:%M%p')
                      if filled else 'pending')
            print(f"  {side:<5} {symbol:<8} {amt:>10} "
                  f"  {status:<10} {date}")
except Exception as e:
    print(f"  Could not load orders: {e}")

# Quick stats on your holdings
if positions:
    print(f"\n  HOLDING STATS")
    print(f"  {'─'*53}")

    tickers_held = [p.symbol for p in positions]
    hist = yf.download(
        tickers_held,
        period='1mo',
        auto_adjust=True,
        progress=False
    )['Close']

    if isinstance(hist, pd.Series):
        hist = hist.to_frame()

    hist   = hist.squeeze(axis=None)
    rets   = hist.pct_change(fill_method=None).dropna()

    for ticker in tickers_held:
        if ticker not in rets.columns:
            continue
        r      = rets[ticker].dropna()
        vol    = r.std() * np.sqrt(252) * 100
        mo_ret = ((1 + r).prod() - 1) * 100
        print(f"  {ticker:<8} "
              f"1mo return: {mo_ret:>+7.2f}%  "
              f"annualized vol: {vol:>5.1f}%")

print(f"\n{'=' * 55}\n")

  PORTFOLIO SUMMARY
  June 17, 2026 04:18 PM

  ACCOUNT
  Total equity              $  103,991.93
  Cash                      $    5,604.00
  Invested                  $   98,387.93
  Today P&L                 ▼ $  1,151.22 (-1.09%)

  POSITIONS (10 stocks)

  Ticker    Shares     Entry   Current      Value    Day %  Total %
  -------- ------- --------- --------- ---------- -------- --------
  HAL        270.7 $   36.53 $   36.53 $    9,889 ▼  1.80% ▲  0.00%
  OXY        184.8 $   53.50 $   53.49 $    9,888 ▼  0.33% ▼  0.01%
  SLB        191.8 $   51.57 $   51.56 $    9,887 ▼  2.84% ▼  0.02%
  BKR        162.6 $   60.83 $   60.81 $    9,886 ▼  1.15% ▼  0.03%
  APA        277.0 $   34.14 $   34.13 $    9,454 ▼  0.38% ▼  0.03%
  PSX         58.7 $  168.57 $  168.51 $    9,885 ▼  2.03% ▼  0.04%
  LVS        203.4 $   48.61 $   48.59 $    9,885 ▼  0.71% ▼  0.04%
  BG          85.4 $  115.79 $  115.71 $    9,882 ▼  3.11% ▼  0.07%
  LDOS        88.3 $  112.00 $  111.76 $    9,868 ▼  1.60% ▼ 

In [14]:
# Mean Reversion Final Results
import numpy as np
import yfinance as yf

# Rebuild signal using existing data
def zscore(prices, window=20):
    mean = prices.rolling(window).mean()
    std  = prices.rolling(window).std()
    return (prices - mean) / std

z = zscore(data)

def mean_rev_weights(z_scores, top_n=10,
                     z_threshold=-1.5):
    oversold = (z_scores < z_threshold).astype(int)
    ranks    = z_scores.rank(axis=1, ascending=True)
    in_top   = (ranks <= top_n).astype(int)
    signal   = oversold * in_top
    weights  = signal.div(
        signal.sum(axis=1), axis=0
    ).fillna(0)
    return weights

weights       = mean_rev_weights(z)
daily_returns = data.pct_change(fill_method=None)
port_returns  = (weights.shift(1) *
                 daily_returns).sum(axis=1)

# SPY
spy     = yf.download('SPY', start='2018-01-01',
                       auto_adjust=True,
                       progress=False)['Close'].squeeze()
spy_ret = spy.pct_change(fill_method=None).squeeze()

# Metrics
def sharpe(r, rf=0.02):
    e = r - rf/252
    return float((e.mean() / e.std()) * np.sqrt(252))

def sortino(r, rf=0.02):
    e        = r - rf/252
    neg      = e[e < 0]
    downside = np.sqrt((neg**2).mean()) * np.sqrt(252)
    return float(e.mean() * 252 / downside)

def max_drawdown(r):
    c = (1 + r).cumprod()
    return float(((c - c.cummax()) / c.cummax()).min())

def cagr(r):
    c = (1 + r).cumprod()
    y = len(r) / 252
    return float(c.iloc[-1] ** (1/y) - 1)

def win_rate(r):
    return float((r > 0).sum() / len(r))

print("=" * 40)
print("MEAN REVERSION — FINAL BACKTEST RESULTS")
print("=" * 40)
print(f"Strategy CAGR:    {cagr(port_returns):.1%}")
print(f"SPY CAGR:         {cagr(spy_ret):.1%}")
print(f"Sharpe Ratio:     {sharpe(port_returns):.2f}")
print(f"Sortino Ratio:    {sortino(port_returns):.2f}")
print(f"Max Drawdown:     {max_drawdown(port_returns):.1%}")
print(f"Win Rate:         {win_rate(port_returns):.1%}")
print(f"SPY Correlation:  "
      f"{port_returns.corr(spy_ret):.2f}")

MEAN REVERSION — FINAL BACKTEST RESULTS
Strategy CAGR:    20.2%
SPY CAGR:         14.7%
Sharpe Ratio:     0.74
Sortino Ratio:    0.70
Max Drawdown:     -70.2%
Win Rate:         52.9%
SPY Correlation:  0.70


In [16]:
def kelly_fraction(returns):
    wr    = float((returns > 0).mean())
    avg_w = float(returns[returns > 0].mean())
    avg_l = float(abs(returns[returns < 0].mean()))
    f     = (wr / avg_l) - ((1 - wr) / avg_w)
    return max(0, min(f / 2, 0.25))

kelly = kelly_fraction(port_returns)
print(f"Kelly fraction: {kelly:.1%}")

Kelly fraction: 25.0%


In [ ]:
# Walk-Forward Validation Cell

def walk_forward_sharpe(returns, split=0.70,
                        freq='monthly'):
    split_idx  = int(len(returns) * split)
    in_sample  = returns[:split_idx]
    out_sample = returns[split_idx:]

    # Use 12 for monthly, 252 for daily
    ann = 12 if freq == 'monthly' else 252

    is_sharpe  = float(
        (in_sample.mean() /
         in_sample.std()) * np.sqrt(ann)
    )
    oos_sharpe = float(
        (out_sample.mean() /
         out_sample.std()) * np.sqrt(ann)
    )
    degradation = ((is_sharpe - oos_sharpe) /
                    abs(is_sharpe)) * 100

    print("=== WALK-FORWARD VALIDATION ===")
    print(f"Training period:      "
          f"{returns.index[0].date()} to "
          f"{returns.index[split_idx].date()}")
    print(f"Test period:          "
          f"{returns.index[split_idx].date()} to "
          f"{returns.index[-1].date()}")
    print(f"In-sample Sharpe:     {is_sharpe:.2f}")
    print(f"Out-of-sample Sharpe: {oos_sharpe:.2f}")
    print(f"Degradation:          {degradation:.1f}%")

    if oos_sharpe > is_sharpe:
        print("✓ Strategy improves out of sample")
    elif degradation < 30:
        print("✓ Acceptable degradation — "
              "strategy is robust")
    else:
        print("⚠ High degradation — "
              "possible overfitting")

walk_forward_sharpe(port_returns, freq='daily')